# Phase 0: Environment Setup

This notebook prepares your environment for the Custom Deep Research Lab.
It verifies cluster access, creates the project namespace, checks the RHOAI operator,
deploys supporting services (PostgreSQL + pgvector), and auto-detects cluster configuration.

## 1. Load Environment

Load variables from `.env`. If no `.env` exists, create one from `sample.env`.

In [ ]:
import os
import shutil
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env") if os.path.basename(os.getcwd()) != "rhoai-custom-research-lab" else ".env"
sample_path = os.path.join(os.path.dirname(env_path), "sample.env") if os.path.dirname(env_path) else "sample.env"

if not os.path.exists(env_path):
    if os.path.exists(sample_path):
        shutil.copy(sample_path, env_path)
        print(f"Created .env from sample.env at: {env_path}")
    else:
        raise FileNotFoundError(f"Neither .env nor sample.env found. Expected at: {env_path}")

load_dotenv(env_path, override=True)
print(f"\u2705 Environment loaded from: {env_path}")

## 2. Verify Cluster Access

Confirm that `oc` CLI is authenticated and can reach the cluster.

In [ ]:
import subprocess

whoami = subprocess.run(["oc", "whoami"], capture_output=True, text=True)
version = subprocess.run(["oc", "version"], capture_output=True, text=True)

if whoami.returncode != 0:
    print(f"\u274c Not logged in to OpenShift cluster.\n{whoami.stderr.strip()}")
    print("\nRun: oc login <cluster-api-url> --username=<user> --password=<pass>")
else:
    print(f"Logged in as: {whoami.stdout.strip()}")
    print(f"\n{version.stdout.strip()}")
    print("\n\u2705 Cluster access verified.")

## 3. Create Project Namespace

Create the lab namespace defined in `NAMESPACE`. Safe to re-run.

In [ ]:
import subprocess, os

namespace = os.getenv("NAMESPACE", "doc-research-lab")

check = subprocess.run(["oc", "get", "namespace", namespace],
                       capture_output=True, text=True)

if check.returncode == 0:
    print(f"\u26a0\ufe0f Namespace '{namespace}' already exists, skipping.")
else:
    result = subprocess.run(
        ["oc", "create", "namespace", namespace, "--dry-run=client", "-o", "yaml"],
        capture_output=True, text=True
    )
    apply = subprocess.run(["oc", "apply", "-f", "-"],
                          input=result.stdout, capture_output=True, text=True)
    if apply.returncode == 0:
        print(f"\u2705 Namespace '{namespace}' created.")
    else:
        print(f"\u274c Failed to create namespace: {apply.stderr.strip()}")

## 4. Verify RHOAI Operator

Check that the Red Hat OpenShift AI operator is installed on the cluster.

In [ ]:
import subprocess

result = subprocess.run(
    ["oc", "get", "csv", "-n", "redhat-ods-operator",
     "-o", "jsonpath={.items[?(@.metadata.name)].metadata.name}"],
    capture_output=True, text=True
)

if result.returncode == 0 and "rhods-operator" in result.stdout:
    csvs = [c for c in result.stdout.split() if "rhods-operator" in c]
    print(f"Installed CSV: {csvs[0] if csvs else result.stdout.strip()}")
    print("\n\u2705 RHOAI operator is installed.")
else:
    print("\u274c RHOAI operator not found.")
    print("Install it from OperatorHub: 'Red Hat OpenShift AI'")
    print(f"\nDebug: {result.stderr.strip() or result.stdout.strip()}")

## 5. Deploy PostgreSQL + pgvector

If `PGVECTOR_HOST=localhost`, use local compose services. Otherwise, deploy on-cluster.

In [ ]:
import subprocess, os

pghost = os.getenv("PGVECTOR_HOST", "localhost")
namespace = os.getenv("NAMESPACE", "doc-research-lab")

if pghost == "localhost":
    print("\u26a0\ufe0f PGVECTOR_HOST=localhost — using local compose services.")
    print("Run `make dev-up` from the project root to start PostgreSQL + pgvector.")
else:
    pgvector_manifest = f"""apiVersion: apps/v1
kind: Deployment
metadata:
  name: pgvector
  namespace: {namespace}
spec:
  replicas: 1
  selector:
    matchLabels:
      app: pgvector
  template:
    metadata:
      labels:
        app: pgvector
    spec:
      containers:
      - name: pgvector
        image: pgvector/pgvector:pg16
        ports:
        - containerPort: 5432
        env:
        - name: POSTGRES_DB
          value: "{os.getenv('PGVECTOR_DB', 'doc_research')}"
        - name: POSTGRES_USER
          value: "{os.getenv('PGVECTOR_USER', 'postgres')}"
        - name: POSTGRES_PASSWORD
          value: "{os.getenv('PGVECTOR_PASSWORD', 'postgres')}"
        volumeMounts:
        - name: pgdata
          mountPath: /var/lib/postgresql/data
      volumes:
      - name: pgdata
        emptyDir: {{}}
---
apiVersion: v1
kind: Service
metadata:
  name: pgvector
  namespace: {namespace}
spec:
  selector:
    app: pgvector
  ports:
  - port: 5432
    targetPort: 5432
"""
    result = subprocess.run(["oc", "apply", "-f", "-"],
                          input=pgvector_manifest, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"\u2705 PostgreSQL + pgvector deployed to namespace '{namespace}'.")
        print(result.stdout.strip())
    else:
        print(f"\u274c Deployment failed:\n{result.stderr.strip()}")

## 6. Update Environment

Auto-detect the cluster's app domain and persist it to `.env`.

In [ ]:
import subprocess, os, re

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env") if os.path.basename(os.getcwd()) != "rhoai-custom-research-lab" else ".env"


def update_env(key: str, value: str):
    """Update or add a key=value in .env file."""
    if not value:
        return
    with open(env_path, "r") as f:
        content = f.read()
    pattern = rf"^#?\s*{key}=.*$"
    replacement = f"{key}={value}"
    if re.search(pattern, content, re.MULTILINE):
        content = re.sub(pattern, replacement, content, count=1, flags=re.MULTILINE)
    else:
        content = content.rstrip("\n") + f"\n{replacement}\n"
    with open(env_path, "w") as f:
        f.write(content)


existing_domain = os.getenv("CLUSTER_DOMAIN")

if existing_domain:
    print(f"\u26a0\ufe0f CLUSTER_DOMAIN already set: {existing_domain}")
else:
    result = subprocess.run(
        ["oc", "get", "ingresses.config", "cluster",
         "-o", "jsonpath={.spec.domain}"],
        capture_output=True, text=True
    )
    if result.returncode == 0 and result.stdout.strip():
        domain = result.stdout.strip()
        update_env("CLUSTER_DOMAIN", domain)
        print(f"\u2705 CLUSTER_DOMAIN={domain} written to .env")
    else:
        print(f"\u274c Could not detect cluster domain.\n{result.stderr.strip()}")
        print("Set CLUSTER_DOMAIN manually in .env.")